# Loan Default Prediction — Logistic Regression
**Dataset:** Kaggle "Give Me Some Credit" (`data.csv`)

**Goal:** Predict probability that a borrower will experience serious delinquency (90+ days past due) in the next 2 years, using logistic regression with an emphasis on interpretability (odds ratios) over raw accuracy.

## 1. Imports

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

## 2. Load Data
`data.csv` has an unnamed first column which is just a row index, so we drop it with `index_col=0`.

In [4]:
data = pd.read_csv("data.csv", index_col=0)
data.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               150000 non-null  int64  
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 10  NumberOfDependents                    146076 non-null  float64
dtypes: fl

In [6]:
data.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

## 3. Clean Data

Before fixing anything, we first **detect** what's wrong with each column. `.describe()` is the fastest way to spot impossible min/max values, and `.value_counts()` is useful for columns that should only contain a small set of valid integers.

### 3.1 Missing Values — Detection

Check null counts across all columns.

In [7]:
data.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

`MonthlyIncome` and `NumberOfDependents` are the only columns with nulls. Next, check whether these columns are skewed — if so, we should impute with median, not mean.

In [8]:
print("MonthlyIncome skew:", data["MonthlyIncome"].skew())
print("NumberOfDependents skew:", data["NumberOfDependents"].skew())

data[["MonthlyIncome", "NumberOfDependents"]].describe()

MonthlyIncome skew: 114.04031794523321
NumberOfDependents skew: 1.5882423788858833


,MonthlyIncome,NumberOfDependents
count,1.202690e+05,146076.000000
mean,6.670221e+03,0.757222
std,1.438467e+04,1.115086
min,0.000000e+00,0.000000
25%,3.400000e+03,0.000000
50%,5.400000e+03,0.000000
75%,8.249000e+03,1.000000
max,3.008750e+06,20.000000


Both columns have a high positive skew (long right tail — a few very high incomes / large families pull the mean up). This confirms **median** is the safer imputation choice, since it isn't dragged by extreme values.

In [9]:
# Impute missing values with median (confirmed skewed distributions above)
data["MonthlyIncome"] = data["MonthlyIncome"].fillna(data["MonthlyIncome"].median())
data["NumberOfDependents"] = data["NumberOfDependents"].fillna(data["NumberOfDependents"].median())

# Confirm no more nulls
data.isnull().sum()

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

### 3.2 Invalid Age — Detection

`.describe()` on `age` shows the min/max, which is the fastest way to catch an impossible value like `age = 0`.

In [10]:
data["age"].describe()

count    150000.000000
mean         52.295207
std          14.771866
min           0.000000
25%          41.000000
50%          52.000000
75%          63.000000
max         109.000000
Name: age, dtype: float64

In [11]:
# How many rows actually have age = 0?
(data["age"] == 0).sum()

np.int64(1)

Minimum age is `0`, which is not a valid age for a credit applicant — this is a data entry error, not a real value. Only a small number of rows are affected, so we replace rather than drop them.

In [12]:
# Fix age=0 (invalid) -> replace with median age
data.loc[data["age"] == 0, "age"] = data["age"].median()

# Confirm
(data["age"] == 0).sum()

np.int64(0)

### 3.3 Sentinel Values in "Late Payment" Columns — Detection

These three columns should logically be small counts (0, 1, 2, 3...). `.value_counts()` reveals whether any value looks out of place.

In [13]:
late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

for col in late_cols:
    print(col)
    print(data[col].value_counts().sort_index())
    print()

NumberOfTime30-59DaysPastDueNotWorse
NumberOfTime30-59DaysPastDueNotWorse
0     126018
1      16033
2       4598
3       1754
4        747
5        342
6        140
7         54
8         25
9         12
10         4
11         1
12         2
13         1
96         5
98       264
Name: count, dtype: int64

NumberOfTime60-89DaysPastDueNotWorse
NumberOfTime60-89DaysPastDueNotWorse
0     142396
1       5731
2       1118
3        318
4        105
5         34
6         16
7          9
8          2
9          1
11         1
96         5
98       264
Name: count, dtype: int64

NumberOfTimes90DaysLate
NumberOfTimes90DaysLate
0     141662
1       5243
2       1555
3        667
4        291
5        131
6         80
7         38
8         21
9         19
10         8
11         5
12         2
13         4
14         2
15         2
17         1
96         5
98       264
Name: count, dtype: int64



Each of these columns has a small number of rows with the value `96` or `98` — a huge jump compared to the realistic range (mostly 0–13). These are known placeholder/error codes in this dataset, not genuine counts of 96 or 98 missed payments. We replace them with the median of the *valid* values (excluding the sentinel rows).

In [14]:
for col in late_cols:
    median_val = data.loc[data[col] < 90, col].median()
    data.loc[data[col] >= 90, col] = median_val

# Confirm sentinel values are gone
for col in late_cols:
    print(col, "max value now:", data[col].max())

NumberOfTime30-59DaysPastDueNotWorse max value now: 13
NumberOfTime60-89DaysPastDueNotWorse max value now: 11
NumberOfTimes90DaysLate max value now: 17


### 3.4 Outliers — Detection

`RevolvingUtilizationOfUnsecuredLines` should realistically be close to 0–1 (it's a ratio/utilization), and `DebtRatio` is also a ratio. `.describe()` shows whether the max is reasonable.

In [15]:
data[["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]].describe()

,RevolvingUtilizationOfUnsecuredLines,DebtRatio
count,150000.000000,150000.000000
mean,6.048438,353.005076
std,249.755371,2037.818523
min,0.000000,0.000000
25%,0.029867,0.175074
50%,0.154181,0.366508
75%,0.559046,0.868254
max,50708.000000,329664.000000


The `max` values for both columns are absurdly large (thousands, when these are supposed to be ratios that are mostly under 1–2). This indicates extreme outliers, likely from data entry or edge-case errors, not realistic borrower behavior. Rather than drop these rows entirely and lose sample size, we **cap** them at the 99.5th percentile.

In [16]:
for col in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]:
    print(col, "99.5th percentile:", data[col].quantile(0.995))

RevolvingUtilizationOfUnsecuredLines 99.5th percentile: 1.3662693040650091
DebtRatio 99.5th percentile: 6186.010000000009


In [17]:
# Cap extreme outliers at 99.5th percentile
for col in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]:
    cap = data[col].quantile(0.995)
    data[col] = np.where(data[col] > cap, cap, data[col])

data[["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]].describe()

,RevolvingUtilizationOfUnsecuredLines,DebtRatio
count,150000.000000,150000.000000
mean,0.322328,325.217939
std,0.356769,955.301614
min,0.000000,0.000000
25%,0.029867,0.175074
50%,0.154181,0.366508
75%,0.559046,0.868254
max,1.366269,6186.010000


## 4. Target Variable

`SeriousDlqin2yrs` is our target. Check class balance — this dataset is known to be imbalanced (most borrowers do NOT default).

In [18]:
data["SeriousDlqin2yrs"].value_counts(normalize=True)

SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

## 5. Train/Test Split

We use `stratify=y` to keep the same class ratio in both train and test sets, since the target is imbalanced.

In [19]:
TARGET = "SeriousDlqin2yrs"

X = data.drop(columns=[TARGET])
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((120000, 10), (30000, 10))

## 6. Scale Features

Logistic regression coefficients are scale-sensitive, so we standardize features (mean 0, std 1) before fitting. We fit the scaler **only on training data** and apply it to test data, to avoid data leakage.

In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. Train Logistic Regression

`class_weight="balanced"` reweights the loss function so the minority class (defaults) isn't ignored, without fabricating synthetic samples like SMOTE would.

In [21]:
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

## 8. Evaluate

For credit risk specifically, accuracy alone is misleading on imbalanced data. We use:
- **AUC-ROC** — overall ranking ability
- **KS statistic** and **Gini coefficient** — standard credit risk industry metrics
- **Confusion matrix** / classification report — precision, recall, F1 per class

In [22]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"AUC-ROC: {auc:.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
ks_stat = max(tpr - fpr)
print(f"KS Statistic: {ks_stat:.4f}")
print(f"Gini Coefficient: {2*auc - 1:.4f}")

AUC-ROC: 0.8586
KS Statistic: 0.5595
Gini Coefficient: 0.7172


In [23]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[22495  5500]
 [  494  1511]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.80      0.88     27995
           1       0.22      0.75      0.34      2005

    accuracy                           0.80     30000
   macro avg       0.60      0.78      0.61     30000
weighted avg       0.93      0.80      0.85     30000



## 9. Coefficient Interpretation (Odds Ratios)

This is the key advantage of logistic regression over black-box models: every feature's effect can be explained directly.

`odds_ratio = exp(coefficient)`
- `odds_ratio > 1` → increases odds of default
- `odds_ratio < 1` → decreases odds of default
- `odds_ratio ≈ 1` → little to no effect

In [24]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0])
}).sort_values("odds_ratio", ascending=False)

coef_df

,feature,coefficient,odds_ratio
0,RevolvingUtilizationOfUnsecuredLines,0.745795,2.108116
6,NumberOfTimes90DaysLate,0.485365,1.624768
2,NumberOfTime30-59DaysPastDueNotWorse,0.393546,1.482227
8,NumberOfTime60-89DaysPastDueNotWorse,0.312900,1.367385
5,NumberOfOpenCreditLinesAndLoans,0.152114,1.164293
7,NumberRealEstateLoansOrLines,0.146419,1.157681
9,NumberOfDependents,0.010232,1.010285
3,DebtRatio,-0.088726,0.915096
4,MonthlyIncome,-0.218106,0.804040
1,age,-0.286199,0.751113
